# Feature experiment: venue-quality signal from OpenAlex

**Question.** `venue` (free-text journal/conference name) is high-cardinality (919
unique values / 2,873 rows), ~27% missing, and some of the longest strings are dirty
data (citation dumps / paper titles that landed in the field by mistake, not real
venue names) — see `notebooks/eda/wf_data_enrich.ipynb`. Raw `citation_count` is also
already known to be a weak/inverted relevance proxy (negative-labelled papers have
*higher* mean citation counts than positive ones in 5 of 6 use cases).

This notebook checks whether an **external, structured venue-quality signal** (via
the free, no-auth [OpenAlex](https://api.openalex.org) `/sources` endpoint) is a
viable replacement/complement: does it resolve cleanly, and — the real test — when
joined onto our own papers, does it track something *different* from citation_count,
or just restate it?

This is a viability check, not a pipeline build — no output is written back to
`data/processed/`.

In [1]:
import json
import re
import time
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)


## 1. Load data

In [2]:
df = pd.read_parquet("../../data/processed/papers_combined.parquet")
print(df.shape)
df[["paper_id", "use_case_key", "venue", "triage_label", "citation_count", "relevance_score"]].head()


(2873, 50)


,paper_id,use_case_key,venue,triage_label,citation_count,relevance_score
0,carbon_capture__3355,carbon_capture,results in engineering,positive,21,0.799504
1,carbon_capture__1143,carbon_capture,the journal of chemical physics,positive,8,0.790661
2,carbon_capture__1588,carbon_capture,,positive,0,0.787967
3,carbon_capture__1621,carbon_capture,,positive,2,0.776255
4,carbon_capture__3493,carbon_capture,carbon capture science & technology,positive,21,0.775960


## 2. Recap: `venue` is high-cardinality and dirty

Confirms the known numbers from `wf_data_enrich.ipynb` before building on them.

In [3]:
venue_missing = df["venue"].isna() | (df["venue"].fillna("") == "")
print(f"rows with no venue recorded: {venue_missing.sum()} / {len(df)} ({venue_missing.mean():.1%})")
print(f"unique venue values (incl. missing/empty): {df['venue'].nunique(dropna=True)}")
print()
print("top 10 venues by frequency:")
print(df["venue"].value_counts().head(10))


rows with no venue recorded: 779 / 2873 (27.1%)
unique venue values (incl. missing/empty): 919

top 10 venues by frequency:
venue
                                       730
arxiv                                   96
scientific reports                      62
materials (basel, switzerland)          36
frontiers in microbiology               31
cement and concrete research            28
construction and building materials     28
nature communications                   26
ssrn electronic journal                 21
arxiv (cornell university)              21
Name: count, dtype: int64


In [4]:
# "Dirty" venues: long strings that look like citation dumps or paper titles,
# not journal/conference names -- used later as a stress test (section 5).
venue_len = df["venue"].str.len().fillna(0)
print(f"venues > 100 chars: {(venue_len > 100).sum()}")
long_venues = df.loc[venue_len > 100, "venue"].drop_duplicates().sort_values(key=lambda s: s.str.len(), ascending=False)
for v in long_venues.head(5):
    print(f"- ({len(v)} chars) {v[:120]}{'...' if len(v) > 120 else ''}")


venues > 100 chars: 48
- (234 chars) thailand concrete association, ed. further reduction of co2 -emissions and circularity in the cement and concrete indust...
- (201 chars) pan, g., zhou, t., zhao, j., li, z., lin, y., ma, b., ... & wang, s. (2026). graph neural networks based analog circuit ...
- (175 chars) proceedings of the 2025 conference of the nations of the americas chapter of the association for computational linguisti...
- (163 chars) enhancing drug repurposing accuracy through ai-orchestrated integration of knowledge graph reasoning, structure-base mod...
- (158 chars) proceedings of the 9th joint sighum workshop on computational linguistics for cultural heritage, social sciences, humani...


## 3. OpenAlex lookup helper

`GET /sources?search=<name>` is a free-text **relevance-ranked search**, not an
exact-name lookup — worth knowing before trusting a "top result" blindly. Two things
found while building this, folded into the helper below:

- Searching `"materials (basel, switzerland)"` as-is returns **zero** results — the
  NLM-style `(City, Country)` qualifier isn't part of OpenAlex's own indexed text.
  Stripping a trailing parenthetical and retrying finds it.
- Searching the stripped `"materials"` alone returns 1,342 matches with *Advanced
  Materials* ranked #1 — the exact `Materials` (the MDPI journal we actually mean) is
  #9. So: prefer an exact (case-insensitive) `display_name` match among the results
  over just taking result #1, and only fall back to the top result if no exact match
  shows up.

One HTTP call per unique venue name, `time.sleep` between calls, each wrapped in
`try/except` so one failed/odd lookup doesn't kill the run — no API key needed.

In [5]:
OPENALEX_SOURCES_URL = "https://api.openalex.org/sources"
USER_AGENT = "TIRI-research/1.0 (mailto:warren.fauvel@gmail.com)"


def _openalex_sources_search(query, per_page=10):
    """One GET to /sources?search=<query>. Returns a list of result dicts (possibly empty)."""
    url = f"{OPENALEX_SOURCES_URL}?search={urllib.parse.quote(query)}&per_page={per_page}"
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=15) as resp:
        return json.load(resp).get("results", [])


def _candidate_queries(venue_name):
    """Yield the venue string as-is, then with a trailing parenthetical stripped
    (handles NLM-style 'Journal (City, Country)' names OpenAlex's search chokes on)."""
    yield venue_name
    stripped = re.sub(r"\s*\([^)]*\)\s*$", "", venue_name).strip()
    if stripped and stripped != venue_name:
        yield stripped


def lookup_venue_quality(venue_name, per_page=25, sleep=0.5):
    """Look up a venue name against OpenAlex /sources. Returns a dict of metrics
    plus how the match was made, or None if every candidate query came back empty."""
    for query in _candidate_queries(venue_name):
        try:
            results = _openalex_sources_search(query, per_page=per_page)
        except Exception as exc:
            print(f"  [error] query={query!r}: {exc}")
            results = []
        time.sleep(sleep)
        if not results:
            continue
        # Compare against the query actually sent this iteration (not the
        # original venue_name) -- on the parenthetical-stripped retry, the
        # thing we're now looking for an exact match on IS the stripped text.
        exact = [r for r in results if r.get("display_name", "").strip().lower() == query.strip().lower()]
        chosen = exact[0] if exact else results[0]
        ss = chosen.get("summary_stats") or {}
        return {
            "venue": venue_name,
            "query_used": query,
            "match_type": "exact" if exact else "top_result_fallback",
            "matched_name": chosen.get("display_name"),
            "works_count": chosen.get("works_count"),
            "cited_by_count": chosen.get("cited_by_count"),
            "mean_citedness_2yr": ss.get("2yr_mean_citedness"),
            "h_index": ss.get("h_index"),
        }
    return None


## 4. Look up the 10 known-clean venues

The 10 highest-frequency, real (non-dirty) venue names in this dataset.

In [6]:
clean_venues = [
    "arxiv",
    "scientific reports",
    "materials (basel, switzerland)",
    "frontiers in microbiology",
    "cement and concrete research",
    "construction and building materials",
    "nature communications",
    "ssrn electronic journal",
    "arxiv (cornell university)",
    "frontiers in plant science",
]

clean_lookups = []
for v in clean_venues:
    result = lookup_venue_quality(v)
    if result is None:
        print(f"[no match] {v!r}")
    else:
        clean_lookups.append(result)

openalex_df = pd.DataFrame(clean_lookups)
openalex_df


,venue,query_used,match_type,matched_name,works_count,cited_by_count,mean_citedness_2yr,h_index
0,arxiv,arxiv,top_result_fallback,arXiv (Cornell University),3276675,7195423,0.180959,693
1,scientific reports,scientific reports,exact,Scientific Reports,306842,7974537,4.753208,462
2,"materials (basel, switzerland)",materials,exact,Materials,57602,1066674,4.289796,245
3,frontiers in microbiology,frontiers in microbiology,exact,Frontiers in Microbiology,41559,1488683,5.191195,344
4,cement and concrete research,cement and concrete research,exact,Cement and Concrete Research,11069,894030,12.572028,368
5,construction and building materials,construction and building materials,exact,Construction and Building Materials,48631,2134376,9.274957,349
6,nature communications,nature communications,exact,Nature Communications,91429,7649091,16.614596,746
7,ssrn electronic journal,ssrn electronic journal,exact,SSRN Electronic Journal,1667612,5451657,0.202507,452
8,arxiv (cornell university),arxiv (cornell university),exact,arXiv (Cornell University),3276675,7195423,0.180959,693
9,frontiers in plant science,frontiers in plant science,exact,Frontiers in Plant Science,35498,1269607,5.166743,326


**Sanity check.** All 10 resolved to a plausible `matched_name` — eyeballing each
query against its match, none look wrong. Two things worth flagging, not hiding:

- `materials (basel, switzerland)` needed the parenthetical-stripping retry to get
  *any* results (0 for the raw string), and the retry's own top-ranked result is the
  wrong journal (`Advanced Materials`, ranked #1 by relevance for query `"materials"`)
  — it only resolves correctly because an **exact** `display_name` match (`Materials`,
  ranked #9) is preferred over the top-ranked one. Without that exact-match
  preference this venue would have silently joined onto the wrong journal's metrics.
- `arxiv` and `arxiv (cornell university)` — two *different* raw strings in our own
  `venue` column — both resolve to the **same** OpenAlex source (`arXiv (Cornell
  University)`, same `works_count`/`h_index`). Our "clean, real" venue list isn't
  even internally deduplicated; a naive exact-string join undercounts the same real
  venue as two.

## 5. Stress test: dirty venue strings (negative-result check)

Three long, clearly-not-a-venue strings from section 2 — a full reference citation,
a proceedings-title dump, and what looks like a paper title — through the exact same
lookup function.

In [7]:
dirty_examples = [
    "pan, g., zhou, t., zhao, j., li, z., lin, y., ma, b., ... & wang, s. (2026). "
    "graph neural networks based analog circuit link prediction. engineering "
    "applications of artificial intelligence, 163, 113035",
    "enhancing drug repurposing accuracy through ai-orchestrated integration of "
    "knowledge graph reasoning, structure-base model prediction, and gene "
    "expression analysis",
    "thailand concrete association, ed. further reduction of co2 -emissions and "
    "circularity in the cement and concrete industry, 16th international congress "
    "on the chemistry of cement 2023 - iccc2023 (bangkok 18.-22.09.2023). bangkok, 2023",
]

for d in dirty_examples:
    result = lookup_venue_quality(d)
    label = "no match (0 results)" if result is None else f"matched -> {result['matched_name']!r} ({result['match_type']})"
    print(f"- ({len(d)} chars) {d[:70]}...\n  {label}\n")


- (201 chars) pan, g., zhou, t., zhao, j., li, z., lin, y., ma, b., ... & wang, s. (...
  no match (0 results)



- (163 chars) enhancing drug repurposing accuracy through ai-orchestrated integratio...
  no match (0 results)



- (234 chars) thailand concrete association, ed. further reduction of co2 -emissions...
  no match (0 results)



**Finding (hard result, all 3 failed the same way).** All three dirty strings return
zero OpenAlex results — the search fails safe rather than silently matching garbage
to some real, wrong venue. That's the good news for lookup *safety*; the bad news is
it also means dirty venue rows simply won't get a venue-quality feature at all,
compounding the ~27% missing-venue rate below.

## 6. Real join: OpenAlex venue-quality metrics onto `papers_combined.parquet`

For each of the 10 clean venues: how many of our papers come from it, how our own
analysts triaged them, and their citation/relevance numbers — joined against the
external venue-quality metrics above.

In [8]:
rows = []
for v in clean_venues:
    sub = df[df["venue"] == v]
    n_reviewed = int(sub["triage_label"].notna().sum())
    rows.append({
        "venue": v,
        "n_papers": len(sub),
        "n_reviewed": n_reviewed,
        "n_positive": int((sub["triage_label"] == "positive").sum()),
        "n_negative": int((sub["triage_label"] == "negative").sum()),
        "n_pass": int((sub["triage_label"] == "pass").sum()),
        "positive_rate": (sub["triage_label"] == "positive").sum() / n_reviewed if n_reviewed else np.nan,
        "mean_citation_count": sub["citation_count"].mean(),
        "n_citation_nonnull": int(sub["citation_count"].notna().sum()),
        "mean_relevance_score": sub["relevance_score"].mean(),
    })

triage_df = pd.DataFrame(rows)
joined = triage_df.merge(openalex_df, on="venue")
joined


,venue,n_papers,n_reviewed,n_positive,n_negative,n_pass,positive_rate,mean_citation_count,n_citation_nonnull,mean_relevance_score,query_used,match_type,matched_name,works_count,cited_by_count,mean_citedness_2yr,h_index
0,arxiv,96,62,28,33,1,0.451613,<NA>,0,0.435975,arxiv,top_result_fallback,arXiv (Cornell University),3276675,7195423,0.180959,693
1,scientific reports,62,29,16,11,2,0.551724,17.083333,36,0.490568,scientific reports,exact,Scientific Reports,306842,7974537,4.753208,462
2,"materials (basel, switzerland)",36,16,7,5,4,0.437500,0.3,10,0.462744,materials,exact,Materials,57602,1066674,4.289796,245
3,frontiers in microbiology,31,31,5,12,14,0.161290,249.0625,16,0.608141,frontiers in microbiology,exact,Frontiers in Microbiology,41559,1488683,5.191195,344
4,cement and concrete research,28,24,17,2,5,0.708333,257.464286,28,0.521124,cement and concrete research,exact,Cement and Concrete Research,11069,894030,12.572028,368
5,construction and building materials,28,21,9,4,8,0.428571,143.321429,28,0.504327,construction and building materials,exact,Construction and Building Materials,48631,2134376,9.274957,349
6,nature communications,26,23,11,11,1,0.478261,599.4,25,0.659068,nature communications,exact,Nature Communications,91429,7649091,16.614596,746
7,ssrn electronic journal,21,21,9,3,9,0.428571,0.428571,21,0.624873,ssrn electronic journal,exact,SSRN Electronic Journal,1667612,5451657,0.202507,452
8,arxiv (cornell university),21,17,16,1,0,0.941176,9.285714,21,0.535335,arxiv (cornell university),exact,arXiv (Cornell University),3276675,7195423,0.180959,693
9,frontiers in plant science,20,20,4,12,4,0.200000,381.882353,17,0.618872,frontiers in plant science,exact,Frontiers in Plant Science,35498,1269607,5.166743,326


**Finding (hard numbers from the join).**
- `arxiv`: 96 papers, but **0 of 96 have a non-null `citation_count`** — per the
  NULL-≠-0 convention this venue's `mean_citation_count` is "never found", not zero,
  and its 96 papers split roughly down the middle on triage (45% positive of 62
  reviewed) despite arXiv's OpenAlex `mean_citedness_2yr` being the lowest of the 10
  (0.18) — a low external prestige score alongside a perfectly ordinary positive rate.
- `cement and concrete research` has both the **highest positive rate (71%, 17/24
  reviewed)** and a mid-table venue-quality score (`mean_citedness_2yr` 12.6, well
  below Nature Communications' 16.6) — our analysts liked this venue's papers far
  more than a pure prestige ranking would predict.
- `frontiers in plant science` (20%) and `frontiers in microbiology` (16%) have the
  **lowest positive rates** of the 10, yet sit mid-table on external venue quality
  (`mean_citedness_2yr` ~5.2 each, similar to `materials`/`scientific reports`) — low
  analyst approval isn't confined to obscure venues either.
- Every venue has papers below `n_papers` with a non-null `citation_count` (e.g.
  `materials (basel, switzerland)`: 10/36) — coverage of citation_count itself is
  uneven even within a single venue, another reason not to lean on it alone.

## 7. Does venue quality look different from `citation_count`, or does it just track it?

The real question for this feature: is `mean_citedness_2yr` telling us something
`citation_count` doesn't, or just restating it? Only 10 venues here, so treat these
as a **directional, exploratory read** (judgement call), not a robust statistical
estimate — the sample is far too small for a confident correlation coefficient.

In [9]:
corr_pairs = [
    ("mean_citedness_2yr", "mean_citation_count", "venue quality vs. this venue's own mean citation_count"),
    ("mean_citedness_2yr", "positive_rate", "venue quality vs. analyst positive rate"),
    ("h_index", "positive_rate", "venue h-index vs. analyst positive rate"),
    ("mean_citation_count", "positive_rate", "mean citation_count vs. analyst positive rate (venue-level echo of the known paper-level finding)"),
]

corr_rows = []
for a, b, label in corr_pairs:
    corr_rows.append({"comparison": label, "x": a, "y": b, "pearson_r (n=10)": joined[a].corr(joined[b])})

pd.DataFrame(corr_rows)


,comparison,x,y,pearson_r (n=10)
0,venue quality vs. this venue's own mean citati...,mean_citedness_2yr,mean_citation_count,0.784196
1,venue quality vs. analyst positive rate,mean_citedness_2yr,positive_rate,-0.038805
2,venue h-index vs. analyst positive rate,h_index,positive_rate,0.466111
3,mean citation_count vs. analyst positive rate ...,mean_citation_count,positive_rate,-0.333859


**Finding (hard numbers, n=10 — read directionally).**
- `mean_citedness_2yr` vs. `mean_citation_count`: **r ≈ 0.78** — a strong positive
  correlation. Venue quality and this venue's own average citation count move
  together closely; they look like largely the *same* axis here, not two
  independent signals.
- `mean_citedness_2yr` vs. `positive_rate`: **r ≈ -0.04** — essentially no
  relationship. External venue prestige does not track how our analysts actually
  triaged papers from that venue, at this venue-level view.
- `mean_citation_count` vs. `positive_rate`: **r ≈ -0.33** — negative, echoing (at
  venue-level aggregation, not the original paper-level or per-use-case analysis)
  the known finding that higher citation counts associate with *lower* positive
  rates, not higher.

Taken together: venue quality mostly **restates** citation_count (r≈0.78) rather
than complementing it, and where it differs from citation_count it's similarly
disconnected from analyst judgement (r≈-0.04 vs. -0.33 — both close to noise, not a
usable signal either way).

## 8. Viability verdict

- **(a) Coverage ceiling.** ~27% of rows have no venue at all, and section 5 shows
  the long-tail dirty strings fail lookup entirely rather than degrading gracefully
  — real achievable coverage is meaningfully below 73%, and shrinks further outside
  this dataset's top 10 venues (919 unique values, most singleton/rare).
- **(b) Lookup cost is genuinely cheap.** One `/sources?search=` call per *unique*
  clean venue name, not per paper — 10 calls resolved every paper from those venues
  in section 6, well under a second each, no API key. This part of the pitch holds up.
- **(c) Complementary signal — the deciding factor — does not hold up here.** Section
  7's join shows venue quality tracks `citation_count` itself (r≈0.78) far more than
  it tracks analyst triage (r≈-0.04, no better than citation_count's own -0.33). It
  isn't offering a materially different read on relevance than the signal it was
  meant to improve on.

**Verdict (judgement call, not a hard metric): not worth building as a Week-3
pipeline feature as-is.** The coverage ceiling alone caps its usefulness, and the one
join that could have justified paying that coverage cost — a venue-quality axis that
diverges usefully from citation_count — didn't show up in this sample. If venue-level
features are revisited later, a normalized/deduplicated venue field (the `arxiv` /
`arxiv (cornell university)` split in section 4 shows the current one isn't even
internally consistent) would need to come first, and a bigger sample of venues than
10 before trusting any correlation number here as more than directional.